# Mapping Sea Surface Temperature using NOAA Data

## Libraries

In [ ]:
### PIP INSTALLS:
# pip install netCDF4 numpy pandas matplotlib plotnine  # run once if needed
###
import datetime
import os

import netCDF4
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

## The Data

### How is environmental data organized?
Earth scientists such as geographers and oceanographers use a special data format specifically designed for Earth science data known as NetCDF (Network Common Data Form). NetCDF is specially suited for spatial data about the Earth because it organizes data points into **layers**. Each layer represents one time step, and contains the longitude, latitude, and data points for one moment in time. Scientists can examine how the data changes over time by navigating between these layers. The Python library **netCDF4** allows us to use and manipulate NetCDF files programmatically.

### How can I access environmental data?

NOAA has several data products monitoring and estimating sea surface temperature. We will use **NOAA OISST v2.1** (Optimum Interpolation SST, also called "Reynolds" SST), which blends satellite and in situ (ship and buoy) measurements into a complete global map of **daily** sea surface temperature at 0.25° (~25 km) resolution. It runs from September 1981 to within a few days of today, so we can compare the present-day ocean to the early 1980s. The data is hosted by NOAA's Physical Sciences Laboratory (PSL), and you can browse it in the [PSL data catalog](https://psl.noaa.gov/data/gridded/data.noaa.oisst.v2.highres.html).

We will access the data through a data server known as THREDDS (Thematic Real-time Environmental Distributed Data Services). THREDDS lets us open a remote NetCDF file over the internet using the **OPeNDAP** protocol and download *only the slice we ask for*, rather than the whole multi-gigabyte file. The server publishes one file per year for each variable:

* `sst.day.mean.YYYY.nc`: daily sea surface temperature (variable `sst`, in °C)
* `sst.day.anom.YYYY.nc`: daily sea surface temperature anomaly (variable `anom`, in °C)

The NetCDF4 **Dataset** function allows us to open a NetCDF file by supplying a path on your local machine or a URL to a data server. After we open the file, we can print it to see the information (metadata) attached to it. This tells us the variable names, coordinates (such as time, latitude, and longitude), as well as other useful information.

Let's start with this year's daily SST file.

In [ ]:
base_url = "https://psl.noaa.gov/thredds/dodsC/Datasets/noaa.oisst.v2.highres"
year = str(datetime.date.today().year)

sst_url = base_url + "/sst.day.mean." + year + ".nc"
sst_nc = netCDF4.Dataset(sst_url)

Printing the connection shows the dataset's metadata. It looks like a lot, but it actually contains useful information! For one, we can see that the dataset contains variables plotted over longitude, latitude, and time. Note the dimensions: 1440 longitudes (0.25° steps), 720 latitudes, and one time step per day so far this year.

In [ ]:
print(sst_nc)

### Finding the most recent day

The **time** variable is stored as "days since 1800-01-01". We can convert it to real dates and find the last layer, which is the most recent day of data. (Data is typically 1-3 days behind the present, and NOAA notes that the most recent ~2 weeks may be revised.)

In [ ]:
time_raw = sst_nc["time"][:]
dates = [datetime.date(1800, 1, 1) + datetime.timedelta(days=int(t)) for t in time_raw]

latest = len(dates) - 1  # Python counts from 0, so the last layer is len(dates) - 1
dates[latest]

Now we can use slices to choose the variable we want to work with: **sst**. Let's set up a variable to hold the latest SST measurements using the date we saved above, for all latitudes and longitudes. Access it using the following pattern:

***dataframe["variablename"][time, lat, lon]***


In [ ]:
# get the latest SST for all lat/lon
sst_latest =  # EDIT HERE

Let's take a look at the variable we just grabbed (sst_latest). What are the dimensions of this variable? (5min explore)

In [ ]:
# Find the dimensions of sst_latest
# YOUR CODE HERE #


What do you think these dimensions could represent? Why is it arranged this way? (Hint: what defines coordinates in a map?)

**Answer**: _________



**sst_latest** is organized into a matrix of size _latitude by longitude_. Each value in our matrix represents the sea surface temperature (°C) at that coordinate point. Land has no value (NA).

**Question**: How are map coordinates typically organized (which comes first: lat or lon)? Do we need to change our dataset's shape to accommodate this?

**Answer**: _________


NetCDF4 orders the dimensions [time, lat, lon], so .T reorders the result to [lon, lat].

We also want to grab the coordinates that the data uses, so that we know what coordinates to give our map.

In [ ]:
#EDIT HERE


In [ ]:
lon = sst_nc["lon"][:]
lat = sst_nc["lat"][:]
print((float(lon.min()), float(lon.max())))
print((float(lat.min()), float(lat.max())))

Notice that longitude runs from 0 to 360 rather than -180 to 180, and that latitude runs from south to north. We'll take care of both when we make our map.

After you are done accessing your variables, it is good practice to close the server connection with **close.nc()**

In [ ]:
sst_nc.close()

Now that we have all the variables we need, let's get to plotting!

## Plotting the Data

To plot our NetCDF data, we first need to convert it to a **raster**. A raster is a spatial grid where each cell represents a pixel on an image. This can be accomplished by calling the raster function and supplying the variable we want to plot, as well as the range of the coordinate values (x-axis for longitude, and y-axis for latitude).

Rasters expect the top row of the image first (the north), and expect each row to be a latitude, so we need to transpose our matrix (remember how to do that?) and flip it north-to-south using **numpy flipud**. We then use **numpy roll** to convert longitude from 0-360 to -180-180 so that the Americas are on the left and the map is centered like a normal world map.

To keep this tidy (and because we'll do it several times), we will write a small function that does all of this for us.

In [ ]:
# What if we skip the transpose, flip, and roll? Each panel adds one step.
raw = sst_latest                   # [lon, lat], straight from the file
transposed = raw.T                 # rows = latitude, columns = longitude
flipped = np.flipud(transposed)    # north at the top
rolled = np.roll(flipped, flipped.shape[1] // 2, axis=1)  # longitude -180 to 180

steps = [(raw, "1. No changes"),
         (transposed, "2. Transpose only"),
         (flipped, "3. Transpose + flip"),
         (rolled, "4. Transpose + flip + roll")]

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, (arr, title) in zip(axes.flat, steps):
    ax.imshow(arr)
    ax.set_title(title)
fig.tight_layout()
plt.show()

with any numpy function, you can learn about it by typing **?**

In [ ]:
#?np.roll

In [ ]:
#?np.flipud

In [ ]:
def to_raster(mat):
    #EDIT HEREr =                                   # transpose: rows = latitude, columns = longitude
    #EDIT HEREr =                          # flip north-to-south so the north is the top row
    #EDIT HEREr =     # rotate longitude from 0-360 to -180-180
    return r


def plot_raster(r, cmap=None, vmin=None, vmax=None, title=None, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 4))
    im = ax.imshow(r, extent=[-180, 180, -90, 90], cmap=cmap, vmin=vmin, vmax=vmax)
    ax.figure.colorbar(im, ax=ax, shrink=0.7)
    ax.set_title(title)
    return ax


r = to_raster(sst_latest)

# use a plot function to plot the raster we just created (Code together)
# YOUR CODE HERE #

plt.show()

If you want to customize your map, you can change the color mapping R uses for each pixel value using the **get_cmap**. Try adding the argument **cmap = get_cmap("Spectral", 100)** to your plot function call. (5 min explore)

In [ ]:
?plt.get_cmap

In [ ]:
# plot the raster again, this time adding the argument for colors
## YOUR CODE HERE ##

plt.show()

Try playing around with this argument and see how changing the cmap name and number changes your map.

In [ ]:
## YOUR CODE HERE ##

plt.show()

matplotlib's **get_cmap** is a nice color collection for making geography maps. The number supplied to the function tells Python how many colors to include. There are many color palettes available in R; find a more comprehensive list [here](https://www.datanovia.com/en/blog/top-r-color-palettes-to-know-for-great-data-visualization/#r-base-color-palettes).

What a nice map! Notice that we didn't import any map images to overlay on the data. Since our dataset contains sea surface temperature around the globe, we can see landmasses due to their lack of data values.

## Sea Surface Temperature Anomalies

Sea surface temperature is a good metric for visualizing the temperature of our oceans. However, SST is a raw measurement that doesn't account for seasonal changes. In the northern hemisphere, we have hotter temperatures in June-August and cooler temperatures in December-February. These natural fluctuations in temperature (called the climatology) make it difficult to visualize how the climate is changing. To solve this problem, scientists calculate a metric known as the Sea Surface Temperature Anomaly. This measurement takes the sea surface temperature at a point in time and compares it to the average temperature at that location for a climatological period (typically 30 years). The OISST anomaly files (`sst.day.anom.YYYY.nc`) let us explore this metric ourselves. The metadata for these files states that the anomaly is relative to the **1971-2000** average.

Because this dataset is so large, we will not download the full dataset. For this notebook, we will compare the same calendar day in the first full year of the data (1982) to the most recent day of data available. Let's write a function that opens the anomaly file for a given date, downloads just that day's layer, and returns it as a raster.

In [ ]:
def get_ssta(date):
    date = pd.Timestamp(date).date()
    url = base_url + "/sst.day.anom." + str(date.year) + ".nc"
    nc = netCDF4.Dataset(url)
    try:
        # Layer number = days since Jan 1 of that year (Python counts from 0)
        idx = (date - datetime.date(date.year, 1, 1)).days

        # Get the variable called "anom" for that single day
        ssta = nc["anom"][idx, :, :].T
    finally:
        nc.close()
    return to_raster(ssta)

Now we can pick our two dates. We use the same month and day for both so that the seasons match.

In [ ]:
date_last = dates[latest]
date_first = datetime.date(1982, date_last.month, date_last.day)  # (error if today is Feb 29; pick another day)

ssta_first = get_ssta(date_first)
ssta_last = get_ssta(date_last)

print(ssta_first.shape, ssta_first.min(), ssta_first.max())

Like the SST dataset, the anomaly is a raster with a value for each 0.25° cell. Let's plot the SSTA for 1982. Blue means colder than the 1971-2000 average, and red means warmer. We use **clamp()** to cap values at +/-5 °C so that a few extreme cells don't wash out the color scale, and we set `zlim` so that both maps share the same scale.

In [ ]:
ssta_cols = LinearSegmentedColormap.from_list(
    "ssta", ["navy", "dodgerblue", "white", "orange", "firebrick"], N=100)

plot_raster(np.clip(ssta_first, -5, 5), vmin=-5, vmax=5, cmap=ssta_cols,
            title=f"SST anomaly (°C): {date_first}")
plt.show()

The colors in this map represent the difference in sea surface temperature at a location to its climatological mean. Because we have so many data points across the world, it can be hard to see exactly where the SSTA is above average (0). Let's plot only the points where SSTA > 0.

In [ ]:
plot_raster((ssta_first > 0).astype(float), title=f"SSTA > 0: {date_first}")
plt.show()

# Number of ocean cells warmer than the climatology, and the fraction of all ocean cells
print((ssta_first > 0).sum())
print((ssta_first > 0).mean())

This gives us an idea for how the sea surface temperature in 1982 differed from the climate normal. Next, let's investigate SSTA for the most recent day of data.

In [ ]:
## FILL IN CODE BELOW ## (code together)

plt.show()
## END OF YOUR CODE ##

In [ ]:

plt.show()

print((ssta_last > 0).sum())
print((ssta_last > 0).mean())

What do you notice about the graphs we have made? How does SSTA for 1982 differ from SSTA now?



## Where to go from here

Great job creating those plots! I hope you have a better understanding of how scientists store environmental data as well as how to access and explore that data. Here are some ideas to keep exploring:

* Pick a different date, or a different year, and compare. Do any well-known climate events (such as El Niño years like 1998, 2016, or 2023) stand out?
* Subtract the two rasters (`ssta_last - ssta_first`) to map how the change between the two dates is distributed.
* Crop the map to a region of interest with **crop(r, extent(xmin, xmax, ymin, ymax))**, such as the Gulf of Mexico or the Great Barrier Reef.
* Pull the full time series at one location (use `count = c(1, 1, -1)` in var.get.nc to get all days for a single grid cell) and plot it.

Although we have created some great maps, we could definitely improve on these. Try adding a nice title and legend to your map, or changing the color scales for SSTA values. If you wanted, you could even try to overlay a world map onto your plot.

## Other NOAA Data

NOAA has petabytes of data available to the public, and much of it is available through THREDDS or similar servers that support subsetting. Some places to look:

* [NOAA PSL THREDDS catalog](https://psl.noaa.gov/thredds/catalog/Datasets/catalog.html): OISST and many other gridded climate datasets, updated regularly.
* [NOAA Coral Reef Watch](https://coralreefwatch.noaa.gov/product/5km/index_5km_dhw.php): daily 5 km global products for coral bleaching, including SST anomaly, Degree Heating Weeks (thermal stress), and Bleaching Alert Area. The [NOAA_DHW dataset on CoastWatch ERDDAP](https://coastwatch.pfeg.noaa.gov/erddap/griddap/NOAA_DHW.html) serves these as NetCDF with data through the present, using URL-based subsetting. This is the direct successor to the thermal stress information CoRTAD used to provide.
* [NCEI THREDDS server](https://www.ncei.noaa.gov/thredds-ocean/catalog.html), [data available via cloud providers](https://www.noaa.gov/nodd/datasets), the [NOAA Data Catalog](https://data.noaa.gov/dataset/), or NOAA's many data portals such as [CoastWatch](https://coastwatch.noaa.gov/cw_html/cwViewer.html).

## Bonus: Coral Reef locations

NOAA's [Deep Sea Coral and Sponge Map Portal](https://www.ncei.noaa.gov/maps/deep-sea-corals/mapSites.htm) contains information on coral and sponge populations around the world. This could be an interesting dataset to pair with our SST data. Here is some code to get you started:

In [ ]:
# Access and download data at https://www.ncei.noaa.gov/maps/deep-sea-corals/mapSites.htm
# Download the data to your machine and change the path below
# NOTE: the download may take a while - try filtering the years to 2010-2026 (or a subset) if it's taking too long
print(os.getcwd())  # check your working directory
deep_sea_corals = pd.read_csv("./deep_sea_corals.csv", low_memory=False)

print(deep_sea_corals.columns.tolist())  # check the column names, including the longitude/latitude columns
deep_sea_corals.head()

In [ ]:
ax = plot_raster(np.clip(ssta_last, -5, 5), vmin=-5, vmax=5, cmap=ssta_cols)
ax.scatter(deep_sea_corals["longitude (degrees_east)"],
           deep_sea_corals["latitude (degrees_north)"], s=0.25, color="black")
plt.show()

Can you make an interesting plot with these datasets? Try it out below! Feel free to use the example as a starting point, and then modify it to create your own plots.

In [ ]:
# example: get years for corals
print(deep_sea_corals["ObservationDate"].head())

# create new column for years in one format : XXXX
# The dates are a mix of "2022" and "2022-05-21", but the year is always the first
# four characters, so .str[:4] works for both formats
deep_sea_corals = deep_sea_corals.assign(
    year=lambda d: d["ObservationDate"].astype(str).str[:4].astype(int))

In [ ]:
deep_sea_corals["year"].head()

In [ ]:
# check the new column with ggplot: how many observations are there each year?
from plotnine import ggplot, aes, geom_bar, labs, theme_minimal

(ggplot(deep_sea_corals, aes(x="year"))
 + geom_bar()
 + labs(title="Deep sea coral and sponge observations per year",
        x="Year", y="Number of observations")
 + theme_minimal())

In [ ]:
# The coordinate columns have long, awkward names. rename() uses new_name = old_name.
# (If you run this cell twice you'll get an error, because the old names are gone.
#  Just re-run the cell that reads the csv first.)
deep_sea_corals = deep_sea_corals.rename(columns={
    "latitude (degrees_north)": "latitude",
    "longitude (degrees_east)": "longitude"})

Let's plot the corals from one year on top of the SSTA map for that same year. There are three steps: (1) `filter()` the corals to one year, (2) download the SSTA map for that year, and (3) plot the map, then add the corals with `points()`. Try it for 2018 first, then change `my_year` to any year from 2000 to 2025 (there are no observations for 2026 yet).

In [ ]:
my_year = 2018

# 1. Keep only the corals observed in my_year
corals_my_year = deep_sea_corals[deep_sea_corals["year"] == my_year]

# 2. Get the SSTA map for my_year, using the same month and day as our earlier maps
my_date = datetime.date(my_year, date_last.month, date_last.day)
ssta_my_year = get_ssta(my_date)

# 3. Plot the SSTA map, then add the corals on top as points
ax = plot_raster(np.clip(ssta_my_year, -5, 5), vmin=-5, vmax=5, cmap=ssta_cols,
                 title=f"{my_year} - {len(corals_my_year)} observations")
ax.scatter(corals_my_year["longitude"], corals_my_year["latitude"], s=0.25, color="black")
plt.show()

To compare several years, we don't want to copy and paste those steps over and over. Instead, we can bundle them into a **function** with one input: the year. Everything that was different for each year (`my_year`) becomes the function's input, called `year_to_plot`.

In [ ]:
def plot_corals_ssta(year_to_plot, ax=None):
    # 1. corals observed in that year
    corals = deep_sea_corals[deep_sea_corals["year"] == year_to_plot]

    # 2. SSTA map for that year
    date = datetime.date(year_to_plot, date_last.month, date_last.day)
    ssta = get_ssta(date)

    # 3. plot the map, then add the corals
    ax = plot_raster(np.clip(ssta, -5, 5), vmin=-5, vmax=5, cmap=ssta_cols,
                     title=f"{year_to_plot} - {len(corals)} observations", ax=ax)
    ax.scatter(corals["longitude"], corals["latitude"], s=0.25, color="black")

Now we can make a map for any year with a single line. `par(mfrow = c(2, 2))` tells R to put the next four plots in a 2 x 2 grid.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

plot_corals_ssta(2000, axes[0, 0])
plot_corals_ssta(2010, axes[0, 1])
plot_corals_ssta(2018, axes[1, 0])
plot_corals_ssta(2025, axes[1, 1])

fig.tight_layout()

**Make it your own:** change the years above, or edit the function to change what the map shows. For example, try filtering to one type of coral with `filter(year == year_to_plot, VernacularNameCategory == "black coral")`, or change the point colors with `col = "red"` in `points()`. Each map downloads a new SSTA layer, so more years will take longer to run.